# 7D Inverse Development
<!--
2026.04.22 DAJones: Created
2026.05.01 DAJones: replace e1 with e7 for common vector among bivector bases.
2026.05.01 DAJones: observe involute description as reversion composed with bit-7-negated
2026.05.01 DAJones: test involute reversion composed with any odd bit negated.
    # another alternative: reversion composed with "odd dimension negation"
    odd_involute = [1]*128
    reversion = [1]*128
    odd_reversion = [1]*128
    for n in range(128):
        m = Grade[ n & 0b01010101 ]
        if 1 == 1&m: odd_involute[n] *= -1
        if Grade[n] in [2,3,6,7]: reversion[n] *= -1
        odd_reversion[n] = odd_involute[n] * reversion[n]
    # does not work

-->

A&S stands for "Abdulkhaev and Shirokov" the authors of a 7D multivector inverse.  
I became aware of this algorithm from view graphs they presented at ENGAGE 2025.  This algorithm is described here:
[Kamron Abdulkhaev at ENGAGE 2025](https://economics.hse.ru/en/gaa/news/1071598907.html?vision=enabled)
This notebook begins with code constructed from those viewgraphs. The algorithm is verified to work in Cl(7) using a home-grown Python package `clif` being used here.

Noticing that the algorithm contains a determinant formula attributed to 
[Acus & Dargys - Inverse of Multivector: Beyond P+Q=5 Threshold](https://arxiv.org/pdf/1712.05204),
I substituted a different determinant version from that paper -- 
one that held the benefit of simplification in a special way.
I produced an *involution version* using this other determinant formula, and verified it in Cl(7).

With little modification, I was able to substitute my *tweak* formulas, 
developed for another 6D inverse, into this 7D program.
This third and last version of a 7D inverse is the ultimate purpose here.

My work using tweak formulas is just an algebraic simplification and optimization of these algorthms.
The main attritution to Kamron Abdulkhaev and Dmitry Shirokov (A&S) is the clever use of involutions which keeps the implementation 7D.
The idea of using a 6D algorithm to solve the 7D problem should be attrituted to Eckhard Hitzer and Steven Sangwine (H&S).
The variety of determinant formulas making any of these algorithms possible is attributed to A. Acus and A. Dargys (A&D).

<pre>
# grades    0 1 2 3 4 5 6 7 8  Shirokov notation
# reverse   + + - - + + - - +  over tilde
# conjugate + - - + + - - + +  over tilde + over hat
# grade_inv + - + - + - + - +  over hat
# ?         + + + + - - - - +  super triangle
</pre>


In [1]:
#=== This cell is for the construction of a 7D -> 6D grade table ===#
# Once the printed result of M[][] here, 
# the next cell defines MT()() in like-fashion.

def comp(x,d=7): return ((1<<d)-1)^x
def vcomp(V): return [comp(v) for v in V]
# 7D vector basis -- A&S call "generators"
M = [[0,comp(0)]]
e = [1,1<<1,1<<2,1<<3,1<<4,1<<5,1<<6]
print( 'e = ', [f'{a:07b}' for a in e] )
# 6D bivector basis - e[0]=e1 is chosen for the "common vector", 
# e[6]=e7 would have been my choice (turns out not to matter)
b = [ e[0] + a for a in e[1:] ]
print( 'b = ', [f'{a:07b}' for a in b] )
M += [ b + vcomp(b) ]
b2 = []
for i in range(len(b)-1):
    for j in range(i+1,len(b)):
        b2 += [b[i]^b[j]]        
M += [ b2 + vcomp(b2) ]
b3 = []
for i in range(len(b)-2):
    for j in range(i+1,len(b)-1):
        for k in range(j+1,len(b)):
            b3 += [b[i]^b[j]^b[k]] 
M += [ b3 + vcomp(b3) ]
b4 = []
for i in range(len(b)-3):
    for j in range(i+1,len(b)-2):
        for k in range(j+1,len(b)-1):
            for l in range(k+1,len(b)):
                b4 += [b[i]^b[j]^b[k]^b[l]] 
M += [ b4 + vcomp(b4) ]
b5 = e[1:]
M += [ b5 + vcomp(b5) ]
M += [ [e[0], comp(e[0])] ]

# use the ordinals in M for computations
# print the elements each bin -- reproduce appendix 2
def label( n ):
    s = 'e'
    for i in range(1,8):
        if 0<((1<<(i-1))&n):
            s += str(i)
    return s
    
print( '\nA 6D Grading of 7D' )
for G in range(7):
    s = str(G) + ' = {'
    for m in M[G]:
        s += label( m ) + ','
    print( s[:-1] + '}' )

print( '\nA Graded Response to 7D' )
for m in M:
    print( m )


e =  ['0000001', '0000010', '0000100', '0001000', '0010000', '0100000', '1000000']
b =  ['0000011', '0000101', '0001001', '0010001', '0100001', '1000001']

A 6D Grading of 7D
0 = {e,e1234567}
1 = {e12,e13,e14,e15,e16,e17,e34567,e24567,e23567,e23467,e23457,e23456}
2 = {e23,e24,e25,e26,e27,e34,e35,e36,e37,e45,e46,e47,e56,e57,e67,e14567,e13567,e13467,e13457,e13456,e12567,e12467,e12457,e12456,e12367,e12357,e12356,e12347,e12346,e12345}
3 = {e1234,e1235,e1236,e1237,e1245,e1246,e1247,e1256,e1257,e1267,e1345,e1346,e1347,e1356,e1357,e1367,e1456,e1457,e1467,e1567,e567,e467,e457,e456,e367,e357,e356,e347,e346,e345,e267,e257,e256,e247,e246,e245,e237,e236,e235,e234}
4 = {e2345,e2346,e2347,e2356,e2357,e2367,e2456,e2457,e2467,e2567,e3456,e3457,e3467,e3567,e4567,e167,e157,e156,e147,e146,e145,e137,e136,e135,e134,e127,e126,e125,e124,e123}
5 = {e2,e3,e4,e5,e6,e7,e134567,e124567,e123567,e123467,e123457,e123456}
6 = {e1,e234567}

A Graded Response to 7D
[0, 127]
[3, 5, 9, 17, 33, 65, 124, 122, 118, 110, 94,

In [2]:
print( [len(m) for m in M] )
print( "compare with row 6 of Pascal's triangle" )
B = [1]
print(B)
Pascal = [B]
for j in range(8):
    B = [ (B+[0])[i]+([0]+B)[i] for i in range(len(B)+1) ]
    print(B)
    Pascal += [B]


[2, 12, 30, 40, 30, 12, 2]
compare with row 6 of Pascal's triangle
[1]
[1, 1]
[1, 2, 1]
[1, 3, 3, 1]
[1, 4, 6, 4, 1]
[1, 5, 10, 10, 5, 1]
[1, 6, 15, 20, 15, 6, 1]
[1, 7, 21, 35, 35, 21, 7, 1]
[1, 8, 28, 56, 70, 56, 28, 8, 1]


In [3]:
dimensions = 7
def bases(): 
    return 1<<dimensions
    
# The grade of an element is so useful, we table it here
Grade = [0]
while len(Grade) < bases():
    Grade += [g+1 for g in Grade]

MT = (
    (0, 127),
    (3, 5, 9, 17, 33, 65, 124, 122, 118, 110, 94, 62),
    (6, 10, 18, 34, 66, 12, 20, 36, 68, 24, 40, 72, 48, 80, 96, 121, 
     117, 109, 93, 61, 115, 107, 91, 59, 103, 87, 55, 79, 47, 31),
    (15, 23, 39, 71, 27, 43, 75, 51, 83, 99, 29, 45, 77, 53, 85, 101, 
     57, 89, 105, 113, 112, 104, 88, 56, 100, 84, 52, 76, 44, 28, 98, 
     82, 50, 74, 42, 26, 70, 38, 22, 14),
    (30, 46, 78, 54, 86, 102, 58, 90, 106, 114, 60, 92, 108, 116, 120, 
     97, 81, 49, 73, 41, 25, 69, 37, 21, 13, 67, 35, 19, 11, 7),
    (2, 4, 8, 16, 32, 64, 125, 123, 119, 111, 95, 63),
    (1, 126)
)

def abdul_signs( V ):
    signs = [1]*128
    for v in V:
        for n in MT[v]:
            signs[n] = -1
    return signs

Signs4   = abdul_signs( [4] )
Signs236 = abdul_signs( [2,3,6] )
Signs145 = abdul_signs( [1,4,5] )
GISigns = [ [1,-1][1 & g] for g in Grade ] # for "grade involution"

#print( Signs4 )
print( "Signs236 = ", Signs236 )
#print( Signs145 )
#print( GISigns )

def abdulkhaev_7d_inverse( U ):
    Up = Signs236 * U
    W = U * Up
    Wc = W * (Signs145 * (W * W))
    Wa = Signs4 * W
    Wac = Wa * (Signs145 * (Wa * Wa))
    Waca = Signs4 * Wac
    y = Up * ((1/3)*Wc + (2/3)*Waca)
    Uy = U * y
    gUy = GISigns * Uy # grade involution of Uy
    DetU = Uy * gUy
    I = (1/DetU.Reg[0]) * y * gUy
    return I

import clif.context as Clif
from clif.multivector import Accum
import clif.util as util
Clif.Cl(7)
U = Accum.random()
I = abdulkhaev_7d_inverse( U )
print('\nProve inverse: 1/U * U = 1?')
util.print_sparse( U * I )


Signs236 =  [1, -1, 1, 1, 1, 1, -1, 1, 1, 1, -1, 1, -1, 1, -1, -1, 1, 1, -1, 1, -1, 1, -1, -1, -1, 1, -1, -1, -1, -1, 1, -1, 1, 1, -1, 1, -1, 1, -1, -1, -1, 1, -1, -1, -1, -1, 1, -1, -1, 1, -1, -1, -1, -1, 1, -1, -1, -1, 1, -1, 1, -1, 1, 1, 1, 1, -1, 1, -1, 1, -1, -1, -1, 1, -1, -1, -1, -1, 1, -1, -1, 1, -1, -1, -1, -1, 1, -1, -1, -1, 1, -1, 1, -1, 1, 1, -1, 1, -1, -1, -1, -1, 1, -1, -1, -1, 1, -1, 1, -1, 1, 1, -1, -1, 1, -1, 1, -1, 1, 1, 1, -1, 1, 1, 1, 1, -1, 1]

Prove inverse: 1/U * U = 1?
  0. 00000000      1.00000000



The construction above confirms for me that what has been accomplished is a clever generalization of involutions that replaces the Hitzer-Sangwine algorithm which uses a 6D inverse to perform a 7D inverse.  What is going on with the Ws looks like one of the Acus-Dargys determinants.
In fact, it is the first one A&D selected for their table 3 summarizing inverse formulas over dimensions.
$$ \frac{1}{3}Wb_{145}(WW)+\frac{2}{3}b_4(b_4(W)b_{145}(b_4(W)b_4(W)) $$
where $b_{145}(W)$ is a particular involution on $W$ which negates elements listed in the table under 1, 4, and 5.
The structure of that table reflects an embedding of 7D in 6D.

Now, consider that there is an equivalent formula which only uses the 145 involution.  
$$ \frac{1}{3}Wb_{145}(WW)+\frac{2}{3}b_{145}(b_{145}(W)b_{145}(b_{145}(W)b_{145}(W)) $$

I make use of the fact that $b_{145}$ is the same as $b_0$ in the A&D environment to good benefit.
But, we should look closer how the involutions have changed for 7D.  I verified that the $MT[2,3,6]$ elements of $W$ are indeed zero; so treating $W$ like a *self-reverse* multivector seems appropriate in this context.  
So we can write $ b_{145}(W) = - b_0(W) $ and compute
$$ -\frac{1}{3}Wb_0(WW)-\frac{2}{3}b_0(b_0(W)b_0(b_0(W)b_0(W)) $$
in good conscience.
The involutions in this formula are much simpler.



Note that there is a point in the algorithm where we have a $V = s + p*e1234567.$
At this point, we need to multiply by $V' = s - p*e1234567$ to get to a scalar.
$V'V = s^2 - p^2(e1234567)^2$ is a scalar alright, but we can't assume the value of $(e1234567)^2$.

Let's consider if the form of the involution which uses e7 instead of e1 has any benefit or offers any insight.

First we distribute the *even space* in 7D over a 6D bivector grading.  
We choose a special vector, e7 here instead of e1.
- 0. (1) scalar
- 1. (6) bivectors containing e7
- 2. (15) the other bivectors (that is: the bivectors in 6D)
- 3. (20) quadravectors containing e7
- 4. (15) the other quadravectors (that is: the quadravectors in 6D)
- 5. (6) hexavectors containing e7
- 6. (1) the 6D pseudoscalar
The *odd space* completes the design, where we include the 7D complement at each "grade". 

Of course the involution that we want is the one which imitates reversion choosing "grades" {2,3,6}.
At this point in our algorithm conversion, this involution is the only one left.

The advantage of this scheme is that our base grading in the even grades is just the underlying 6D even elements (which of course are still even in 7D).  The base grading in the odd grades is seen to be just the 6D odd elements made even by the the inclusion of `e7`.

In [4]:
# alternate involution
table = []
for n in range(7):
    table += [[]]
for n in range(1<<7):
    g = Grade[n]
    if g in [0,2,4,6]: 
        if 0 == 0b100_0000 & n:
            table[g] += [n]
        else:
            table[g-1] += [n]

alt236 = [1]*128
elements = table[2]+table[3]+table[6]
for e in elements:
    alt236[e] *= -1
    alt236[0b111_1111 ^ e] *= -1
    
print( '\nAn Alternate 6D Grading of 7D' )
for n in range(7):
    s = str(n) + ' = {'
    for m in table[n]:
        s += label( m ) + ','
    for m in table[n]:
        s += label( 0b111_1111 ^ m ) + ','
    print( s[:-1] + '}' )



An Alternate 6D Grading of 7D
0 = {e,e1234567}
1 = {e17,e27,e37,e47,e57,e67,e23456,e13456,e12456,e12356,e12346,e12345}
2 = {e12,e13,e23,e14,e24,e34,e15,e25,e35,e45,e16,e26,e36,e46,e56,e34567,e24567,e14567,e23567,e13567,e12567,e23467,e13467,e12467,e12367,e23457,e13457,e12457,e12357,e12347}
3 = {e1237,e1247,e1347,e2347,e1257,e1357,e2357,e1457,e2457,e3457,e1267,e1367,e2367,e1467,e2467,e3467,e1567,e2567,e3567,e4567,e456,e356,e256,e156,e346,e246,e146,e236,e136,e126,e345,e245,e145,e235,e135,e125,e234,e134,e124,e123}
4 = {e1234,e1235,e1245,e1345,e2345,e1236,e1246,e1346,e2346,e1256,e1356,e2356,e1456,e2456,e3456,e567,e467,e367,e267,e167,e457,e357,e257,e157,e347,e247,e147,e237,e137,e127}
5 = {e123457,e123467,e123567,e124567,e134567,e234567,e6,e5,e4,e3,e2,e1}
6 = {e123456,e7}


Before we leave alt236 as an involution, let's give it a serious description.
First we need a description of the grading:

|    |                        |                      |
| :- | :--------------------  | :------------------  |
| 0. |      scalar            | 6D pseudoscalar & e7 |
| 1. | 6D   vectors    & e7   | 6D 5-vectors         |
| 2. | 6D 2-vectors           | 6D 4-vectors    & e7 |
| 3. | 6D 3-vectors    & e7   | 6D 3-vectors         |
| 4. | 6D 4-vectors           | 6D 2-vectors    & e7 |
| 5. | 6D 5-vectors    & e7   | 6D   vectors         |
| 6. | 6D pseudoscalar        |                   e7 |

So, the elements we negate for the alt236 involution is:

|    |                        |                      |
|:---|:-----------------------|:---------------------|
| 2. | 6D 2-vectors           | 6D 4-vectors    & e7 |
| 3. | 6D 3-vectors    & e7   | 6D 3-vectors         |
| 6. | 6D pseudoscalar        |                   e7 |

Or, considering the first half is 6D, we can describe alt236 as:
grades 2,3,6 from the first half
grades 1,4,5 from the second half

Yet another way to describe involution by direct product is:
modify the standard 7D reversion involution direct-product by negating the 2nd half.


In [5]:
Talt = [[1,1,-1,-1][3&Grade[i]] for i in range(1<<7)]
for i in range(1<<6,len(Talt)): Talt[i] *= -1
for i in range(len(Talt)):
    print(f"{i:07b} {alt236[i]} {Talt[i]}")

0000000 1 1
0000001 1 1
0000010 1 1
0000011 -1 -1
0000100 1 1
0000101 -1 -1
0000110 -1 -1
0000111 -1 -1
0001000 1 1
0001001 -1 -1
0001010 -1 -1
0001011 -1 -1
0001100 -1 -1
0001101 -1 -1
0001110 -1 -1
0001111 1 1
0010000 1 1
0010001 -1 -1
0010010 -1 -1
0010011 -1 -1
0010100 -1 -1
0010101 -1 -1
0010110 -1 -1
0010111 1 1
0011000 -1 -1
0011001 -1 -1
0011010 -1 -1
0011011 1 1
0011100 -1 -1
0011101 1 1
0011110 1 1
0011111 1 1
0100000 1 1
0100001 -1 -1
0100010 -1 -1
0100011 -1 -1
0100100 -1 -1
0100101 -1 -1
0100110 -1 -1
0100111 1 1
0101000 -1 -1
0101001 -1 -1
0101010 -1 -1
0101011 1 1
0101100 -1 -1
0101101 1 1
0101110 1 1
0101111 1 1
0110000 -1 -1
0110001 -1 -1
0110010 -1 -1
0110011 1 1
0110100 -1 -1
0110101 1 1
0110110 1 1
0110111 1 1
0111000 -1 -1
0111001 1 1
0111010 1 1
0111011 1 1
0111100 1 1
0111101 1 1
0111110 1 1
0111111 -1 -1
1000000 -1 -1
1000001 1 1
1000010 1 1
1000011 1 1
1000100 1 1
1000101 1 1
1000110 1 1
1000111 -1 -1
1001000 1 1
1001001 1 1
1001010 1 1
1001011 -1 -1
1001100 1 

In [6]:
# another alternative: reversion composed with "odd dimension negation"
odd_involute = [1]*128
reversion = [1]*128
odd_reversion = [1]*128
for n in range(128):
    m = Grade[ n & 0b01010101 ]
    if 1 == 1&m: odd_involute[n] *= -1
    if Grade[n] in [2,3,6,7]: reversion[n] *= -1
    odd_reversion[n] = odd_involute[n] * reversion[n]
# does not work


In [7]:
# This is just a variation of A&S 7D inverse using a different A&D formula for the determinant.

VERBOSE = True

def K(W): # just change the sign on scalar and pseudoscalar
    V = W.copy()
    V.Reg[0] *= -1
    V.Reg[127] *= -1
    return V    

def L(W): 
    return W * K( W * W )

def also_7d_inverse( U ):
    # Up = Signs236 * U # the originial, using e1 special after H&S
    Up = alt236 * U # the alternate involution template, using e7 instead of e1.
    W = U * Up
    #Wa = K( W )
    #Wb = K( Wa * K( Wa * Wa ) )
    #Wc = W * K( W * W )
    #I = Wc + Wb + Wb
    I = L(W) + 2*K(L(K(W)))
    V = I * W
    # IW = V so 1/W = I/V : because this reciprocal is what we care about 
    # it does not matter if I and V contain the common factor -1/3 
    # needed to express the determinant
    # V is all zero except for scalar and pseudoscalar parts
    Vp = GISigns * V # use the grade involution to obtain something like a conjugate in the complex sense
    D = Vp * V # is now a scalar
    #util.printSparceAccum( D )
    d = D.Reg[0]
    # D = V'V so 1/V = V'/D
    # remember 1/W = I/V so 1/W = IV'/D
    rW = (1/d) * I * Vp # this is 1/W
    # remember W = UUp so 1/U = Up/W
    rU = Up * rW
    if VERBOSE:
        print('\nValidate that we have found rW = 1/W, by printing W * rW' )
        util.print_sparse( W * rW )
        print('\nValidate that we have found rU = 1/U, by printing U * rU' )
        util.print_sparse( U * rU )
    return rU

U = Accum.random()
rU = also_7d_inverse( U )




Validate that we have found rW = 1/W, by printing W * rW
  0. 00000000      1.00000000

Validate that we have found rU = 1/U, by printing U * rU
  0. 00000000      1.00000000


Let $[W]$ mean "just the scalar and pseudoscalar parts of $W$", then $b(W)=W-2[W]$ negates those parts.
$$b(W^2)=W^2-2[W^2]$$ 
So, it's fair to say that the expression above is *reduced*: we can accept our determinant algorithm in terms of $W^n$ and $[W^n]$.
For instance:
$$(b(W))^2=(W-2[W])^2=W^2-4W[W]+4[W]$$
This strategy looks good since it tracks with the 6D work, and that means we can just write down the key ingredient for the determinant formula.  Here is the $S = S(W)$ function satisfying: $-3 det(W) = S*W$
$$S(W) = Wb(W^2) + 2b( b(W) b( (b(W))^2 ) )$$ in terms of $W^n$ and $[W^n]$ can be written:
$$S(W) = 3W^3 - 12[W]W^2 + 24[W]^2W - 6[W^2]W - 4[W^3] + 24[W][W^2] - 32[W]^3$$
Following the 6D example, we successively factor:
$$S_1 = (W-4[W])W,$$
$$S_0 = (S_1-2[S_1])W$$
$$-\frac{1}{3}det(W) = (S_0-\frac{4}{3}[S_0])W$$


In [8]:
# This is the "tweak version" of the 7D inverse

VERBOSE = True

# we turn the negation function into a tweak function
def K( W, s=1 ): 
    V = W.copy()
    V.Reg[0] *= -s
    V.Reg[127] *= -s
    return V

def another_7d_inverse( U ):
    #Up = Signs236 * U # as given by A&S using e1 after H&S
    Up = alt236 * U # the A&S reversion-like involution using e7 instead of e1 as special
    W = U * Up
    I1 = K(  W,  3 ) * W
    I0 = K( I1,  1 ) * W
    I  = K( I0, 1/3)
    V = I * W
    # IW = V so 1/W = I/V : because this reciprocal is what we care about 
    # it does not matter if I and V contain the common factor -1/3 
    # needed to express the determinant
    # V is all zero except for scalar and pseudoscalar parts
    Vp = GISigns * V # use the grade involution to obtain something like a conjugate in the complex sense
    D = Vp * V # is now a scalar
    d = D.Reg[0]
    # D = V'V so 1/V = V'/D
    # remember 1/W = I/V so 1/W = IV'/D
    rW = (1/d) * I * Vp # this is 1/W
    # remember W = UUp so 1/U = Up/W
    rU = Up * rW
    if VERBOSE:
        print('\nValidate that we have found rW = 1/W, by printing W * rW' )
        util.print_sparse( W * rW )
        print('\nValidate that we have found rU = 1/U, by printing U * rU' )
        util.print_sparse( U * rU )
    return rU

import clif.context as Clif
Clif.Cl(7)
U = Accum.random()
rU = another_7d_inverse( U )



Validate that we have found rW = 1/W, by printing W * rW
  0. 00000000      1.00000000

Validate that we have found rU = 1/U, by printing U * rU
  0. 00000000      1.00000000


Here is an important version of 7D inverse.
When we borrow the `Signs236` involution, we can use the next level of real tweaks.
The *complex tweaks* in the A&S like version let us by with fewer multivector products instead of compound tweaks.
That makes the other version likely to be faster, but it's special cased.
This version is interesting in its amphibious nature between FLS and Jones.

One gets the impression that if the right extention of `Signs236` to 8D were known it would just work.


In [9]:
import clif.context as Clif
Clif.Cl(7)
A = Accum.random()

#Z = Signs236
Z = alt236

def I7(A):
    '''returns the inverse of a 7D multivector'''
    V = Z * A
    B = A * V
    G = B.copy()
    G.Reg[0] *= -7
    G *= B
    G.Reg[0] *= -3
    G *= B
    G.Reg[0] *= -5/3
    G *= B
    G.Reg[0] *= -1
    G *= B
    G.Reg[0] *= -3/5
    G *= B
    G.Reg[0] *= -1/3
    G *= B
    G.Reg[0] *= -1/7
    I = G.copy()
    G *= B # G is now a scalar
    # print(G)
    # 1 = B(I/G) = A(~AI/G)
    return V*I.scale(1/G.Reg[0])

I = I7(A)
print(I*A)

00000000       1.00000000



In [10]:
import numpy
import clif.context as Clif
from clif.multivector import Accum

TOLERANCE = 0.000000001

def near( a, b ):
    d = a - b
    verdict = -TOLERANCE < d and d < TOLERANCE
    return verdict

def verify_inverse( A, I ):
    R = A*I
    if not near( R.Reg[0], 1.0 ): 
        return False
    for r in R.Reg[1:]:
        if not near( r, 0.0 ): 
            return False
    return True

samples = numpy.random.normal( 0.0, 1.0, 1<<7 )
signatures = [(p, q, 7-p-q) for p in range(8) for q in range(8-p)]
VERBOSE = False
for s in signatures:
    Clif.Cl(*s) # initialize signature
    msg = f"Cl{s}: "
    for n in range(Clif.dimensions):
        A = Accum()
        A.Reg[1<<n] = 1.0 # e_n
        C = A*A
        c = C.Reg[0]
        msg += f"e{n+1}*e{n+1} = {int(c):2d}, "
    A.Reg = numpy.array(samples).tolist()
    I = another_7d_inverse( A )
#    print(I*A)
    if verify_inverse( A, I ):
        msg += "passed"
    else:
        msg += "failed"
    print( msg )


Cl(0, 0, 7): e1*e1 =  0, e2*e2 =  0, e3*e3 =  0, e4*e4 =  0, e5*e5 =  0, e6*e6 =  0, e7*e7 =  0, passed
Cl(0, 1, 6): e1*e1 = -1, e2*e2 =  0, e3*e3 =  0, e4*e4 =  0, e5*e5 =  0, e6*e6 =  0, e7*e7 =  0, passed
Cl(0, 2, 5): e1*e1 = -1, e2*e2 = -1, e3*e3 =  0, e4*e4 =  0, e5*e5 =  0, e6*e6 =  0, e7*e7 =  0, passed
Cl(0, 3, 4): e1*e1 = -1, e2*e2 = -1, e3*e3 = -1, e4*e4 =  0, e5*e5 =  0, e6*e6 =  0, e7*e7 =  0, passed
Cl(0, 4, 3): e1*e1 = -1, e2*e2 = -1, e3*e3 = -1, e4*e4 = -1, e5*e5 =  0, e6*e6 =  0, e7*e7 =  0, passed
Cl(0, 5, 2): e1*e1 = -1, e2*e2 = -1, e3*e3 = -1, e4*e4 = -1, e5*e5 = -1, e6*e6 =  0, e7*e7 =  0, passed
Cl(0, 6, 1): e1*e1 = -1, e2*e2 = -1, e3*e3 = -1, e4*e4 = -1, e5*e5 = -1, e6*e6 = -1, e7*e7 =  0, passed
Cl(0, 7, 0): e1*e1 = -1, e2*e2 = -1, e3*e3 = -1, e4*e4 = -1, e5*e5 = -1, e6*e6 = -1, e7*e7 = -1, passed
Cl(1, 0, 6): e1*e1 =  1, e2*e2 =  0, e3*e3 =  0, e4*e4 =  0, e5*e5 =  0, e6*e6 =  0, e7*e7 =  0, passed
Cl(1, 1, 5): e1*e1 =  1, e2*e2 = -1, e3*e3 =  0, e4*e4 =  0, e5*

In [11]:

import clif.context as Clif
from clif.multivector import Accum
import I7_instrumented as I7I
import clif.util as util

def show_clifford_environment():
    msg = ""
    for n in range(Clif.dimensions):
        A = Accum()
        A.Reg[1<<n] = 1.0 # e_n
        C = A*A
        c = C.Reg[0]
        msg += f"e{n+1}*e{n+1} = {int(c):2d}, "
    return msg

Clif.Cl(7) # initialize signature

Areg = [0.0]*128
Areg[0b0000000] =  2.0 # scalar
Areg[0b0000001] = -2.0 # e1
Areg[0b0100000] =  1.0 # e6
Areg[0b0010010] = -3.0 # e25
Areg[0b0010101] = -1.0 # e135
Areg[0b0111100] =  1.0 # e3456
Areg[0b1110110] = -2.0 # e23567
Areg[0b1111011] =  1.0 # e124567
Areg[0b1111111] =  3.0 # e1234567

divider = "\n========================================================================="

print( divider, "\nTEST #1: Euclidean" )
Clif.Cl(7)
print( "Cl(7): ", show_clifford_environment() )
A = Accum()
A.Reg = Areg
iA = I7I.I7_instrumented( A )

print( divider, "\nTEST #2: Anti-Euclidean" )
Clif.Cl(0,7)
print( "Cl(0,7): ", show_clifford_environment() )
A = Accum()
A.Reg = Areg
iA = I7I.I7_instrumented( A )

print( divider, "\nTEST #3: Grassman" )
Clif.Cl(0,0,7)
print( "Cl(0,0,7): ", show_clifford_environment() )
A = Accum()
A.Reg = Areg
iA = I7I.I7_instrumented( A )

print( divider, "\nTEST #4: Mixed" )
Clif.Cl(2,3,2)
print( "Cl(2,3,2): ", show_clifford_environment() )
A = Accum()
A.Reg = Areg
iA = I7I.I7_instrumented( A )

print( divider, "\nTEST #5: Conformal" )
Clif.Cl(6,0,1)
print( "Cl(6,0,1): ", show_clifford_environment() )
A = Accum()
A.Reg = Areg
iA = I7I.I7_instrumented( A )



TEST #1: Euclidean
Cl(7):  e1*e1 =  1, e2*e2 =  1, e3*e3 =  1, e4*e4 =  1, e5*e5 =  1, e6*e6 =  1, e7*e7 =  1, 

U =  2 - (2^e1) - (3^e25) - (1^e135) + (1^e6) + (1^e3456) - (2^e23567) + (1^e124567) + (3^e1234567)

Up = Signs236 * U =  2 + (2^e1) + (3^e25) - (1^e135) + (1^e6) + (1^e3456) - (2^e23567) + (1^e124567) + (3^e1234567)

W = U * Up =  4 - (6^e3) + (6^e123) - (12^e14) + (4^e134) - (12^e125) - (4^e135) + (4^e6) + (4^e16) - (2^e146) + (6^e2346) + (4^e3456) + (6^e127) + (4^e2357) - (6^e123457) - (6^e2467) + (2^e23467) - (8^e23567) - (8^e123567) + (4^e24567) + (4^e124567) + (12^e1234567)

I1 = K(  W,  3 ) * W =  96 - (72^e12) + (72^e3) + (96^e13) - (24^e123) + (176^e4) + (272^e14) - (176^e134) - (48^e5) - (48^e15) + (96^e125) - (160^e135) + (32^e145) - (96^e2345) + (128^e6) + (32^e16) - (48^e126) - (112^e146) - (72^e2346) - (56^e156) - (48^e2456) + (48^e3456) + (24^e7) - (136^e127) - (96^e137) + (32^e2347) + (96^e157) - (224^e2357) + (64^e23457) + (16^e123457) + (176^e2467) - (288^